# Eksplorasi Data
**UAS Data Warehouse 2025/2026 | S1 Sains Data UNESA**

## Load Semua Tabel

In [1]:
import pandas as pd
import numpy as np

fact        = pd.read_csv('data/processed/fact_harga_pangan.csv')
fact_ind    = pd.read_csv('data/processed/fact_indikator_ekonomi.csv')
dim_waktu   = pd.read_csv('data/processed/dim_waktu.csv')
dim_negara  = pd.read_csv('data/processed/dim_negara.csv')
dim_komoditas = pd.read_csv('data/processed/dim_komoditas.csv')
dim_indikator = pd.read_csv('data/processed/dim_indikator.csv')

for name, df in [
    ('fact_harga_pangan',       fact),
    ('fact_indikator_ekonomi',  fact_ind),
    ('dim_waktu',               dim_waktu),
    ('dim_negara',              dim_negara),
    ('dim_komoditas',           dim_komoditas),
    ('dim_indikator',           dim_indikator),
]:
    print(f"  {name:30s}: {len(df):,} baris | {len(df.columns)} kolom")

  fact_harga_pangan             : 3,834 baris | 10 kolom
  fact_indikator_ekonomi        : 18 baris | 5 kolom
  dim_waktu                     : 24 baris | 6 kolom
  dim_negara                    : 6 baris | 4 kolom
  dim_komoditas                 : 152 baris | 3 kolom
  dim_indikator                 : 4 baris | 3 kolom


## Validasi Skema

In [2]:
wb_cols_lama = ['fp_cpi_totl_zg', 'ny_gdp_mktp_kd_zg', 'ny_gdp_pcap_cd', 'tm_val_food_zs_un']
found = [c for c in wb_cols_lama if c in fact.columns]
if found:
    print(f"Kolom WB lama masih ada: {found}")
else:
    print("fact_harga_pangan bersih dari kolom WB lama")
expected_fact = ['waktu_id','negara_id','komoditas_id','avg_price','min_price','max_price',
                 'record_count','unit','currency','tahun_partisi']
missing = [c for c in expected_fact if c not in fact.columns]
print(f"Kolom fact_harga_pangan OK" if not missing else f"Kolom hilang: {missing}")

expected_ind = ['waktu_id','negara_id','indikator_id','indicator_value','tahun_partisi']
missing_ind = [c for c in expected_ind if c not in fact_ind.columns]
print(f"Kolom fact_indikator_ekonomi OK" if not missing_ind else f"Kolom hilang: {missing_ind}")
print(f"dim_negara punya kolom iso3" if 'iso3' in dim_negara.columns else "dim_negara tidak punya iso3")
print(f"dim_waktu punya kolom semester" if 'semester' in dim_waktu.columns else "dim_waktu tidak punya semester")

fact_harga_pangan bersih dari kolom WB lama
Kolom fact_harga_pangan OK
Kolom fact_indikator_ekonomi OK
dim_negara punya kolom iso3
dim_waktu punya kolom semester


In [3]:
checks = [
    (fact,     'waktu_id',     dim_waktu,    'waktu_id',    'fact_harga_pangan → dim_waktu'),
    (fact,     'negara_id',    dim_negara,   'negara_id',   'fact_harga_pangan → dim_negara'),
    (fact,     'komoditas_id', dim_komoditas,'komoditas_id','fact_harga_pangan → dim_komoditas'),
    (fact_ind, 'waktu_id',     dim_waktu,    'waktu_id',    'fact_indikator_ekonomi → dim_waktu'),
    (fact_ind, 'negara_id',    dim_negara,   'negara_id',   'fact_indikator_ekonomi → dim_negara'),
    (fact_ind, 'indikator_id', dim_indikator,'indikator_id','fact_indikator_ekonomi → dim_indikator'),
]

print("FK Integrity Check")
for fact_df, fk_col, dim_df, pk_col, label in checks:
    orphan = ~fact_df[fk_col].isin(dim_df[pk_col])
    status = "OK" if orphan.sum() == 0 else f"{orphan.sum()} orphan"
    print(f"  {label:45s}: {status}")

FK Integrity Check
  fact_harga_pangan → dim_waktu                : OK
  fact_harga_pangan → dim_negara               : OK
  fact_harga_pangan → dim_komoditas            : OK
  fact_indikator_ekonomi → dim_waktu           : OK
  fact_indikator_ekonomi → dim_negara          : OK
  fact_indikator_ekonomi → dim_indikator       : OK


## Eksplorasi fact_harga_pangan

In [4]:
print("SPARSITY fact_harga_pangan")
measure_cols = ['avg_price','min_price','max_price','record_count']
total_cells = fact.shape[0] * len(measure_cols)
null_cells  = fact[measure_cols].isnull().sum().sum()
print(f"Total sel measure : {total_cells:,}")
print(f"Sel kosong (NaN)  : {null_cells:,}")
print(f"Sparsity          : {null_cells/total_cells*100:.2f}%")
print()
print("Null per kolom:")
print(fact.isnull().sum())

SPARSITY fact_harga_pangan
Total sel measure : 15,336
Sel kosong (NaN)  : 0
Sparsity          : 0.00%

Null per kolom:
waktu_id         0
negara_id        0
komoditas_id     0
unit             0
currency         0
avg_price        0
min_price        0
max_price        0
record_count     0
tahun_partisi    0
dtype: int64


In [5]:
print("KELENGKAPAN KOMBINASI")
n_waktu     = len(dim_waktu)
n_negara    = len(dim_negara)
n_komoditas = len(dim_komoditas)
theoretical = n_waktu * n_negara * n_komoditas
actual      = len(fact)
coverage    = actual / theoretical * 100
print(f"Kombinasi teoritis (waktu×negara×komoditas): {theoretical:,}")
print(f"Baris aktual                               : {actual:,}")
print(f"Coverage                                   : {coverage:.2f}%")
print(f"Sparsity kombinasi                         : {100-coverage:.2f}%")
print()

KELENGKAPAN KOMBINASI
Kombinasi teoritis (waktu×negara×komoditas): 21,888
Baris aktual                               : 3,834
Coverage                                   : 17.52%
Sparsity kombinasi                         : 82.48%



In [6]:
print("DISTRIBUSI PER NEGARA")
dist = (fact.groupby('negara_id').size()
            .reset_index(name='count')
            .merge(dim_negara[['negara_id','country_name','iso3']], on='negara_id')
            .assign(pct=lambda x: (x['count']/x['count'].sum()*100).round(2))
            .sort_values('count', ascending=False))
print(dist[['country_name','iso3','count','pct']].to_string(index=False))
print(f"\nRatio max/min: {dist['count'].max()/dist['count'].min():.2f}x")

DISTRIBUSI PER NEGARA
country_name iso3  count   pct
 Philippines  PHL   1416 36.93
    Cambodia  KHM   1018 26.55
     Lao PDR  LAO    528 13.77
 Timor-Leste  TLS    429 11.19
   Indonesia  IDN    282  7.36
     Myanmar  MMR    161  4.20

Ratio max/min: 8.80x


In [7]:
print("DISTRIBUSI PER KOMODITAS")
komdist = (fact.groupby('komoditas_id').size()
               .reset_index(name='count')
               .merge(dim_komoditas[['komoditas_id','commodity_name','category']], on='komoditas_id')
               .sort_values('count', ascending=False))
print("Top 10 komoditas terbanyak:")
print(komdist.head(10)[['commodity_name','category','count']].to_string(index=False))
print("\nBottom 10 komoditas tersedikit:")
print(komdist.tail(10)[['commodity_name','category','count']].to_string(index=False))

DISTRIBUSI PER KOMODITAS
Top 10 komoditas terbanyak:
            commodity_name              category  count
                      Eggs   meat, fish and eggs     96
            Meat (chicken)   meat, fish and eggs     94
                    Garlic vegetables and fruits     51
                      Taro    cereals and tubers     48
            Sweet potatoes    cereals and tubers     48
Meat (beef, first quality)   meat, fish and eggs     48
          Tomatoes (local) vegetables and fruits     47
                Oil (palm)          oil and fats     47
    Fuel (petrol-gasoline)              non-food     47
                      Salt    miscellaneous food     47

Bottom 10 komoditas tersedikit:
           commodity_name              category  count
       Chili (red, large) vegetables and fruits      5
      Rice (high quality)    cereals and tubers      5
                     Rice    cereals and tubers      5
    Oil (vegetable, bulk)          oil and fats      5
Chili (bird's eye, gree

In [8]:
print("OUTLIER avg_price")
q1_val = fact['avg_price'].quantile(0.25)
q3_val = fact['avg_price'].quantile(0.75)
iqr    = q3_val - q1_val
lower  = q1_val - 1.5 * iqr
upper  = q3_val + 1.5 * iqr
outliers = fact[(fact['avg_price'] < lower) | (fact['avg_price'] > upper)]
print(f"Q1          : {q1_val:,.4f}")
print(f"Q3          : {q3_val:,.4f}")
print(f"IQR         : {iqr:,.4f}")
print(f"Batas bawah : {lower:,.4f}")
print(f"Batas atas  : {upper:,.4f}")
print(f"Outlier     : {len(outliers):,} dari {len(fact):,} baris ({len(outliers)/len(fact)*100:.2f}%)")
print()
print("DISTRIBUSI avg_price")
print(fact['avg_price'].describe().round(4))
print(f"Skewness : {fact['avg_price'].skew():.4f}")
print(f"Kurtosis : {fact['avg_price'].kurtosis():.4f}")
print()

OUTLIER avg_price
Q1          : 77.8467
Q3          : 8,604.6524
IQR         : 8,526.8057
Batas bawah : -12,712.3618
Batas atas  : 21,394.8610
Outlier     : 563 dari 3,834 baris (14.68%)

DISTRIBUSI avg_price
count      3834.0000
mean      11664.1250
std       25311.1644
min           0.1773
25%          77.8467
50%        1621.8418
75%        8604.6524
max      143772.6131
Name: avg_price, dtype: float64
Skewness : 3.1790
Kurtosis : 10.2170



In [9]:
print("DISTRIBUSI TEMPORAL")
waktu_dist = (fact.groupby('waktu_id').size()
                  .reset_index(name='count')
                  .merge(dim_waktu[['waktu_id','periode','semester']], on='waktu_id')
                  .sort_values('waktu_id'))
print(waktu_dist[['periode','semester','count']].to_string(index=False))

DISTRIBUSI TEMPORAL
periode  semester  count
2024-01         1    177
2024-02         1    179
2024-03         1    179
2024-04         1    179
2024-05         1    179
2024-06         1    160
2024-07         2    161
2024-08         2    160
2024-09         2    160
2024-10         2    160
2024-11         2    161
2024-12         2    160
2025-01         1    106
2025-02         1    114
2025-03         1    160
2025-04         1    160
2025-05         1    161
2025-06         1    160
2025-07         2    159
2025-08         2    160
2025-09         2    159
2025-10         2    160
2025-11         2    160
2025-12         2    160


In [10]:
print("DISTRIBUSI SEMESTER")
sem_dist = (fact.merge(dim_waktu[['waktu_id','year','semester']], on='waktu_id')
                .groupby(['year','semester']).size()
                .reset_index(name='count'))
print(sem_dist.to_string(index=False))

DISTRIBUSI SEMESTER
 year  semester  count
 2024         1   1053
 2024         2    962
 2025         1    861
 2025         2    958


## Eksplorasi fact_indikator_ekonomi

In [11]:
print("RINGKASAN fact_indikator_ekonomi")
fi = (fact_ind
      .merge(dim_waktu[['waktu_id','year']], on='waktu_id')
      .merge(dim_negara[['negara_id','country_name','iso3']], on='negara_id')
      .merge(dim_indikator[['indikator_id','indicator_code','indicator_name']],
             on='indikator_id'))

print(f"Total baris: {len(fi)}")
print(f"Tahun      : {sorted(fi['year'].unique())}")
print(f"Negara     : {sorted(fi['iso3'].unique())}")
print(f"Indikator  : {sorted(fi['indicator_code'].unique())}")

RINGKASAN fact_indikator_ekonomi
Total baris: 18
Tahun      : [np.int64(2024)]
Negara     : ['IDN', 'KHM', 'LAO', 'MMR', 'PHL']
Indikator  : ['FP.CPI.TOTL.ZG', 'NY.GDP.MKTP.KD.ZG', 'NY.GDP.PCAP.CD', 'TM.VAL.FOOD.ZS.UN']


In [12]:
print("COVERAGE PER INDIKATOR × NEGARA")
pivot_coverage = (fi.groupby(['indicator_code','iso3'])
                    .size()
                    .reset_index(name='count')
                    .pivot(index='indicator_code', columns='iso3', values='count')
                    .fillna(0).astype(int))
print(pivot_coverage)
print()

COVERAGE PER INDIKATOR × NEGARA
iso3               IDN  KHM  LAO  MMR  PHL
indicator_code                            
FP.CPI.TOTL.ZG       1    1    1    0    1
NY.GDP.MKTP.KD.ZG    1    1    1    1    1
NY.GDP.PCAP.CD       1    1    1    1    1
TM.VAL.FOOD.ZS.UN    1    1    0    1    1



In [13]:
print("NILAI INDIKATOR PER NEGARA")
pivot_nilai = (fi.pivot_table(
    index='country_name',
    columns='indicator_code',
    values='indicator_value',
    aggfunc='mean'
).round(2))
print(pivot_nilai.to_string())

NILAI INDIKATOR PER NEGARA
indicator_code  FP.CPI.TOTL.ZG  NY.GDP.MKTP.KD.ZG  NY.GDP.PCAP.CD  TM.VAL.FOOD.ZS.UN
country_name                                                                        
Cambodia                  0.81               5.98         2627.88               7.46
Indonesia                 2.18               5.03         4925.43              11.86
Lao PDR                  23.13               4.13         2123.98                NaN
Myanmar                    NaN              -0.97         1359.05               9.74
Philippines               3.21               5.69         3984.83              15.01


In [14]:
print("STATISTIK TIAP INDIKATOR")
for code_val, group in fi.groupby('indicator_code'):
    unit = group['unit_of_measure'].iloc[0] if 'unit_of_measure' in group.columns else '-'
    print(f"\n{code_val} ({unit})")
    print(group['indicator_value'].describe().round(4))

STATISTIK TIAP INDIKATOR

FP.CPI.TOTL.ZG (-)
count     4.0000
mean      7.3332
std      10.5776
min       0.8080
25%       1.8381
50%       2.6971
75%       8.1921
max      23.1306
Name: indicator_value, dtype: float64

NY.GDP.MKTP.KD.ZG (-)
count    5.0000
mean     3.9711
std      2.8531
min     -0.9720
25%      4.1302
50%      5.0303
75%      5.6920
max      5.9751
Name: indicator_value, dtype: float64

NY.GDP.PCAP.CD (-)
count       5.0000
mean     3004.2342
std      1438.3749
min      1359.0500
25%      2123.9791
50%      2627.8797
75%      3984.8315
max      4925.4305
Name: indicator_value, dtype: float64

TM.VAL.FOOD.ZS.UN (-)
count     4.0000
mean     11.0159
std       3.2112
min       7.4595
25%       9.1685
50%      10.7974
75%      12.6448
max      15.0095
Name: indicator_value, dtype: float64


## Harga vs Indikator Ekonomi

In [15]:
fact_enriched = (
    fact
    .merge(dim_waktu[['waktu_id','year','month']], on='waktu_id')
    .merge(dim_negara[['negara_id','country_name','iso3']], on='negara_id')
)

fi_pivot = (fi.pivot_table(
    index=['iso3','year'],
    columns='indicator_code',
    values='indicator_value',
    aggfunc='mean'
).reset_index())
fi_pivot.columns.name = None
fact_enriched = fact_enriched.merge(fi_pivot, on=['iso3','year'], how='left')

avg_tahunan = (fact_enriched
               .groupby(['iso3','country_name','year'])
               .agg(
                   avg_price=('avg_price','mean'),
                   **{c: (c,'mean') for c in fi_pivot.columns if c not in ['iso3','year']}
               )
               .round(2)
               .reset_index())

print("Rata-rata Harga + Indikator Ekonomi per Negara per Tahun")
print(avg_tahunan.to_string(index=False))

Rata-rata Harga + Indikator Ekonomi per Negara per Tahun
iso3 country_name  year  avg_price  FP.CPI.TOTL.ZG  NY.GDP.MKTP.KD.ZG  NY.GDP.PCAP.CD  TM.VAL.FOOD.ZS.UN
 IDN    Indonesia  2024   41837.86            2.18               5.03         4925.43              11.86
 IDN    Indonesia  2025   43380.77             NaN                NaN             NaN                NaN
 KHM     Cambodia  2024    5816.58            0.81               5.98         2627.88               7.46
 KHM     Cambodia  2025    5889.53             NaN                NaN             NaN                NaN
 LAO      Lao PDR  2024   48209.29           23.13               4.13         2123.98                NaN
 LAO      Lao PDR  2025   49665.50             NaN                NaN             NaN                NaN
 MMR      Myanmar  2024    4845.73             NaN              -0.97         1359.05               9.74
 MMR      Myanmar  2025    4727.55             NaN                NaN             NaN                Na

In [16]:
print("Korelasi avg_price vs Indikator Ekonomi")
ind_cols = [c for c in avg_tahunan.columns
            if c.startswith(('FP.','NY.','TM.'))]

if ind_cols:
    corr = avg_tahunan[['avg_price'] + ind_cols].corr()['avg_price'].drop('avg_price')
    print(corr.round(4))
    print()
    print("Korelasi rendah bisa disebabkan perbedaan skala mata uang antar negara.")
else:
    print("Tidak ada data indikator yang bisa dikorelasikan.")

Korelasi avg_price vs Indikator Ekonomi
FP.CPI.TOTL.ZG       0.6522
NY.GDP.MKTP.KD.ZG    0.1525
NY.GDP.PCAP.CD       0.2119
TM.VAL.FOOD.ZS.UN    0.0481
Name: avg_price, dtype: float64

Korelasi rendah bisa disebabkan perbedaan skala mata uang antar negara.


## Ringkasan Limitasi Data

In [17]:
print("LIMITASI DATA")
print()
print("fact_harga_pangan:")
print(f"  - {len(fact):,} baris | 6 negara | 2024–2025")
negara_list = dim_negara['country_name'].tolist()
print(f"  - Negara: {negara_list}")
print(f"  - Coverage kombinasi: {len(fact)/(len(dim_waktu)*len(dim_negara)*len(dim_komoditas))*100:.2f}%")
print()
print("fact_indikator_ekonomi:")
print(f"  - {len(fact_ind):,} baris | hanya tahun 2024 (Opsi A)")
print("  - MMR tidak punya data CPI")
print("  - LAO dan TLS tidak punya data food import")
print()

LIMITASI DATA

fact_harga_pangan:
  - 3,834 baris | 6 negara | 2024–2025
  - Negara: ['Cambodia', 'Indonesia', 'Lao PDR', 'Myanmar', 'Philippines', 'Timor-Leste']
  - Coverage kombinasi: 17.52%

fact_indikator_ekonomi:
  - 18 baris | hanya tahun 2024 (Opsi A)
  - MMR tidak punya data CPI
  - LAO dan TLS tidak punya data food import

